# Solutions · 06 · Long-run behaviour and mixing

Worked solutions to the exercises of [notebook 06](../notebooks/06_long_run_behaviour_and_mixing.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engstoch import markov as mk, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Exercise 1

For the lazy random walk on a cycle of n vertices compute the mixing time t_mix(1/4) for n = 8..128 and show that it grows like n^2. Compare with the relaxation time 1/gap.

In [2]:
rows = []
for n in (8, 16, 32, 64, 128):
    P = np.zeros((n, n))
    for i in range(n):
        P[i, i] = 0.5; P[i, (i + 1) % n] += 0.25; P[i, (i - 1) % n] += 0.25
    c = mk.MarkovChain(P)
    rows.append((n, c.mixing_time(0.25), c.spectral_gap()["relaxation_time"]))
r = np.array(rows, float)
print(pd.DataFrame(rows, columns=["n", "t_mix(1/4)", "relaxation time"]).to_string(index=False, float_format="%.1f"))
print("t_mix / n^2:", (r[:, 1] / r[:, 0] ** 2).round(4).tolist(), " relaxation / n^2:", (r[:, 2] / r[:, 0] ** 2).round(4).tolist())

  n  t_mix(1/4)  relaxation time
  8           6              6.8
 16          25             26.3
 32          98            104.1
 64         389            415.3
128        1555           1660.4
t_mix / n^2: [0.0938, 0.0977, 0.0957, 0.095, 0.0949]  relaxation / n^2: [0.1067, 0.1026, 0.1016, 0.1014, 0.1013]


Both grow like n²: a random walk needs about n² steps to diffuse a distance n (spectral gap 1 − cos(2π/n) ≈
2π²/n² for the lazy walk, halved). Unlike the Ehrenfest urn there is no log factor - the cycle has no cutoff.
Diffusive mixing is the reason local MCMC moves are slow on long, thin targets.

## Exercise 2

Find the stationary law of the (s, S) inventory chain for every s in 0..4 and S in s+1..10 with holding cost 1 per unit per week, lost sale cost 20 and ordering cost 15. Which policy minimises the long-run weekly cost?

In [3]:
def weekly_cost(s, S, h=1.0, lost=20.0, order=15.0, mean=2.0):
    m = models.inventory_chain(s, S, mean); c = m["chain"]; pmf = m["demand_pmf"]
    pi = c.stationary(); k = np.arange(len(pmf))
    cost = 0.0
    for x in range(S + 1):
        start = S if x <= s else x
        cost += pi[x] * ((order if x <= s else 0.0) + lost * np.sum(pmf * np.maximum(k - start, 0)) + h * x)
    return cost
grid = [(s, S, weekly_cost(s, S)) for s in range(0, 5) for S in range(s + 1, 11)]
best = min(grid, key=lambda t: t[2])
print(f"best policy s = {best[0]}, S = {best[1]}: {best[2]:.3f} per week")
print("a few others:", [(s, S, round(float(c), 3)) for s, S, c in grid if (s, S) in ((2, 6), (0, 4), (3, 8), (4, 10))])

best policy s = 2, S = 10: 9.014 per week
a few others: [(0, 4, 15.241), (2, 6, 10.225), (3, 8, 9.758), (4, 10, 10.28)]


The weekly cost is the stationary expectation of ordering cost (when stock ends at or below s), expected lost
sales given the starting stock, and holding cost on the end-of-week stock. A full grid of policies is cheap
because each needs one linear solve. The best policy reorders at s = 2 but orders up to S = 10, the largest
value on the grid: with an ordering cost of 15 against a holding cost of 1 per unit-week, large infrequent
orders pay, and the grid should be extended before trusting the optimum.

## Exercise 3

**Implement it yourself:** write power iteration for PageRank on a sparse link list (dict of out-links), with dangling pages and damping, and reproduce `markov.pagerank` on the web graph.

In [4]:
links = {0: [1, 2], 1: [2, 3], 2: [0, 3, 4], 3: [4, 5], 4: [2, 5], 5: []}
n, d = 6, 0.85
r = np.full(n, 1 / n)
for it in range(200):
    new = np.full(n, (1 - d) / n)
    dangling = sum(r[i] for i in links if not links[i])
    new += d * dangling / n
    for i, outs in links.items():
        for j in outs:
            new[j] += d * r[i] / len(outs)
    if np.max(np.abs(new - r)) < 1e-14:
        break
    r = new
print(f"converged in {it} iterations:", r.round(6).tolist())
print("library:                   ", mk.pagerank(models.web_graph()["adjacency"]).round(6).tolist())

converged in 39 iterations: [0.118007, 0.10376, 0.227292, 0.162105, 0.186901, 0.201935]
library:                    [0.118007, 0.10376, 0.227292, 0.162105, 0.186901, 0.201935]


Only the links are stored; each iteration distributes every page's rank over its out-links, spreads the
dangling pages' rank uniformly, and adds the teleportation share. The error contracts by at least the factor d
per iteration, so 0.85ᵏ < 10⁻¹⁴ needs about 200 iterations at worst - and much fewer in practice.